# Mission 2 — clean 59D vs eGeMAPSv02 88D

**구현만 수행 / 실제 학습·비교 미실행. 실행과 해석은 사용자 소유.**

동일 classifier·평가 계약에서 representation 교체의 효과를 비교한다. 사용자가 고른 clean 기준은 Accuracy **0.8737597742437785**, CM `[[72265,13086],[9371,83169]]`다. old-cache 0.874558 및 HCE 2377은 재현 기준이 아니다.

Training 내부 fixed split만 사용한다. extraction·Official Validation 경로가 없고 fit gate 기본값은 `False`다. checkpoint를 위에서 아래로 실행한다. 실패하면 다음 fit으로 넘어가지 않는다. 재실행은 새 kernel에서 시작한다. 기존 cache·실험로그·roadmap을 수정하지 않는다.

## 1. Config / environment

처음에는 `RUN_MODEL_FIT=False`로 로드·계약을 확인한다. 그 뒤 사용자가 명시적으로 `True`로 바꿔 학습한다. 새로운 실행 ID를 매번 사용하며 기존 결과를 덮어쓰지 않는다.

In [5]:
from pathlib import Path
from datetime import datetime, timedelta, timezone
from uuid import uuid4
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "src/dcc_data").is_dir():
    raise RuntimeError("STOP: repository root not found")
sys.path.insert(0, str(REPO_ROOT / "src"))
from dcc_data import mission2_standard_representation_comparison as comparison
from dcc_data.mission2_acoustic_feature_expansion import M04_ITER100_FEATURE_SCREENING_CONFIG
RUN_MODEL_FIT = True
CHAIN_ROOT = REPO_ROOT / "data/processed/mission2_stage4_clean/stage4_clean_v1_090422f783f0d911"
EGEMAPS_PATH = REPO_ROOT / "data/processed/mission2_egemaps_colab_v1/training_egemaps_v02_88d.npz"
EXPERIMENT_ID = "clean59d_egemaps88d_" + datetime.now(timezone(timedelta(hours=9))).strftime("%Y%m%dT%H%M%S") + "_" + uuid4().hex[:8]
OUTPUT_ROOT = REPO_ROOT / "data/processed/mission2_standard_representation_comparison" / EXPERIMENT_ID
ENVIRONMENT = comparison.environment_info(REPO_ROOT)
print(ENVIRONMENT)
print("M04-100:", dict(M04_ITER100_FEATURE_SCREENING_CONFIG))
print("Cache:", EGEMAPS_PATH)
print("Output:", OUTPUT_ROOT)
print("Threshold >=0.5; scaler internal_train_only")

{'timestamp': '2026-10-04T19:01:53.669037+09:00', 'python_executable': 'c:\\Users\\YW\\anaconda3\\envs\\dcc-prelim\\python.exe', 'python': '3.12.7', 'numpy': '2.5.2', 'sklearn': '1.9.0', 'pandas': '3.0.5', 'git_branch': 'yeonwoo/mission2-standard-representation', 'git_commit': '3a873810b1cea8434a90393d18ebdf7381002772', 'git_status': 'M notebooks/mission2_standard_representation.ipynb\n M src/dcc_data/mission2_standard_representation.py\n M tests/test_mission2_model_experiments.py\n M tests/test_mission2_standard_representation.py\n?? .tmp.driveupload/\n?? .vscode/\n?? docs/mission2_experiment_log.md\n?? docs/superpowers/\n?? notebooks/mission2_egemaps_colab_resume.ipynb\n?? notebooks/mission2_standard_representation_comparison.ipynb\n?? notebooks/tmp/\n?? scripts/mission2_egemaps_transfer_identity.py\n?? src/dcc_data/mission2_clean_stage4.py\n?? src/dcc_data/mission2_egemaps_colab.py\n?? src/dcc_data/mission2_egemaps_resume.py\n?? src/dcc_data/mission2_standard_representation_comparis

## 2. Cache integrity / provenance

strict loader로 기존 cache만 읽는다. clean chain의 모든 artifact hash와 source 계약도 확인하므로 시간이 걸릴 수 있다. shape·dtype·finite·feature 순서·metadata·fingerprint·padded 3행 계약 위반은 STOP한다.

NPZ에는 label 배열이 없다. Colab transfer manifest SHA256과 로컬 manifest SHA256을 검증하고 manifest label을 clean 59D label과 비교한다. key 일치만으로 raw waveform 내용의 동일성을 확정하지 않는다. clean의 content-proof 상태와 Colab overlap·approval 기록은 summary에 보존한다.

In [2]:
clean, egemaps, X88, tracking, alignment = comparison.load_inputs(REPO_ROOT, EGEMAPS_PATH, CHAIN_ROOT)
assert clean.X.shape == (889358, 59) and X88.shape == (889358, 88)
assert clean.X.dtype == X88.dtype == np.float32
assert np.isfinite(clean.X).all() and np.isfinite(X88).all()
print("59D/88D:", clean.X.shape, X88.shape)
print("Fingerprint:", clean.provenance_fingerprint)
print("Padded:", egemaps.metadata["padded_row_count"])
print("Unscaled matrix bytes:", clean.X.nbytes, X88.nbytes)

59D/88D: (889358, 59) (889358, 88)
Fingerprint: 49d65a35d1e98c280f98e45cbed864ef8bac36268ccffbe81aa0985923c90de3
Padded: 3
Unscaled matrix bytes: 209888488 313054016


## 3. Record-key alignment

unique key set 일치 뒤 deterministic alignment를 수행한다. missing·extra·duplicate·label·call mismatch가 하나라도 있으면 STOP한다. tracking은 model feature와 분리한다.

In [3]:
assert alignment["status"] == "PASS"
assert np.array_equal(tracking.record_key.to_numpy(), clean.record_keys)
assert np.array_equal(tracking.call_id.to_numpy(), clean.call_ids)
assert np.array_equal(tracking.speaker.to_numpy(), clean.y)
display(pd.DataFrame([alignment]))

,status,rows,missing,extra,duplicate_keys,label_mismatches,call_mismatches,label_source,reordered_88d
0,PASS,889358,0,0,0,0,0,transfer-manifest SHA256 + clean cache labels,False


## 4. Fixed split / scaling

기존 seed 42 GroupSplit helper를 재사용한다. 711,467/177,891행, 23,360/5,840 calls, call overlap 0을 검증한다. 두 scaler는 각각 internal train에만 fit한다. split key/index hash를 결과와 함께 기록한다.

In [4]:
prepared59, prepared88, split_audit = comparison.prepare_pair(clean, egemaps, X88)
assert np.array_equal(prepared59.train_idx, prepared88.train_idx)
assert np.array_equal(prepared59.validation_idx, prepared88.validation_idx)
assert prepared59.scaler.n_samples_seen_ == prepared88.scaler.n_samples_seen_ == 711467
assert not set(clean.call_ids[prepared59.train_idx]) & set(clean.call_ids[prepared59.validation_idx])
display(pd.DataFrame([split_audit]))

,seed,test_size,total_rows,train_rows,validation_rows,train_calls,validation_calls,call_overlap,train_keys_hash,validation_keys_hash,train_idx_sha256,validation_idx_sha256
0,42,0.2,889358,711467,177891,23360,5840,0,1e297c22fff7fef9d4b8cf6f1319b526e3e7097373e3bf...,d7a387b986bbc3b037e1db2df67a4e8dfba91727a2186d...,c6b7602c181367c8dfe2260064db977b3795d5632b3fc4...,7a3b9ac54da179e1e5f7d01672a1b181a923b5cff4013d...


## 5. 59D reproduction — 첫 학습 checkpoint

`RUN_MODEL_FIT=True`일 때만 학습한다. clean 기준 CM의 정수 하나라도 달라지면 REPRODUCTION_STOP하며 88D를 학습하지 않는다. Accuracy는 소수점 끝자리 수준만 허용한다.

과거 row prediction artifact가 없으므로 CM/Accuracy 일치만으로 모든 과거 prediction의 동일성을 주장하지 않는다. 이번 reference prediction과 지표는 별도로 저장한다. 재현 실패 결과도 보존하고 STOP한다.

In [6]:
result59 = comparison.fit_prepared(prepared59, allow_fit=RUN_MODEL_FIT)
report59, diagnostics59 = comparison.evaluate_result(result59, prepared59, tracking)
try:
    reproduction = comparison.verify_reproduction(result59)
    reproduction_error = None
except RuntimeError as error:
    reproduction = {"status": "REPRODUCTION_STOP", "reason": str(error)}
    reproduction_error = error
reference_predictions = pd.DataFrame({
    "record_key": result59.fit_audit["validation_record_keys"],
    "call_id": clean.call_ids[prepared59.validation_idx], "y_true": result59.y_true,
    "pred_59d": result59.y_pred, "prob0_59d": result59.probabilities[:, 0],
    "prob1_59d": result59.probabilities[:, 1], "confidence_59d": result59.probabilities.max(axis=1),
})
comparison.save_artifacts(OUTPUT_ROOT / "reference", {
    "experiment_id": EXPERIMENT_ID, "environment": ENVIRONMENT,
    "source_fingerprint": clean.provenance_fingerprint, "chain_path": CHAIN_ROOT,
    "split": split_audit, "model_config": dict(result59.model_config),
    "report59": report59, "reproduction": reproduction,
}, reference_predictions, pd.DataFrame([report59]), pd.DataFrame())
display(pd.DataFrame([report59]))
print(reproduction)
if reproduction_error is not None:
    raise reproduction_error

,sample_count,accuracy,confusion_matrix,recall_0,recall_1,recall_speaker_0,recall_speaker_1,correct_count,representation,dimension,...,convergence_warning,warnings,fit_seconds,validation_inference_seconds,train_inference_seconds,scaled_matrix_bytes,scaler_fit_scope,feature_names_hash,prediction_hash,fit_array_hashes
0,177891,0.87376,"[[72265, 13086], [9371, 83169]]",0.84668,0.898736,0.84668,0.898736,155434,AF10_AF11_COMBINATION,59,...,True,[Stochastic Optimizer: Maximum iterations (100...,189.472715,0.119767,0.319432,209888488,internal_train_only,7f411fb583c526c56f1d52e3df2e4947b012d3b2fef412...,d65f39ff6eaf1b068a8de855cfddf8465ba7ae6458acbe...,{'split_fingerprint': 'e5b03c2133ace054d313cbe...


{'status': 'REPRODUCED_METRICS', 'expected_accuracy': 0.8737597742437785, 'expected_confusion_matrix': [[72265, 13086], [9371, 83169]], 'historical_row_prediction_equality': 'UNVERIFIABLE: historical row predictions unavailable', 'accuracy_float_tolerance': 1e-12}


## 6. 88D fit / evaluation

동일 M04-100·seed·평가 행·0.5 threshold를 유지한다. helper에서도 59D 재현과 row identity를 재검사한다. tuning은 하지 않는다.

In [7]:
if reproduction["status"] != "REPRODUCED_METRICS":
    raise RuntimeError("REPRODUCTION_STOP: reference must pass first")
result88 = comparison.fit_candidate(prepared88, result59, allow_fit=RUN_MODEL_FIT)
report88, diagnostics88 = comparison.evaluate_result(result88, prepared88, tracking)
assert dict(result59.model_config) == dict(result88.model_config)
display(pd.DataFrame([report88]))

,sample_count,accuracy,confusion_matrix,recall_0,recall_1,recall_speaker_0,recall_speaker_1,correct_count,representation,dimension,...,convergence_warning,warnings,fit_seconds,validation_inference_seconds,train_inference_seconds,scaled_matrix_bytes,scaler_fit_scope,feature_names_hash,prediction_hash,fit_array_hashes
0,177891,0.86399,"[[70299, 15052], [9143, 83397]]",0.823646,0.901199,0.823646,0.901199,153696,EGEMAPSV02_FUNCTIONALS_88D,88,...,True,[Stochastic Optimizer: Maximum iterations (100...,186.145962,0.404899,0.378094,313054016,internal_train_only,c54e0a9dc9e8e63d4cac0afc35d508e7652e2fb872576c...,0b4821ba3e1e0cf366cec15c9154a19b99d810180e0bb5...,{'split_fingerprint': 'e5b03c2133ace054d313cbe...


## 7. Paired / duration / overlap diagnostics

같은 177,891행에서 correction·regression·net과 both correct/wrong을 집계한다. speaker, overlap, 요청한 5초 duration bins, 두 weak slice를 함께 본다. 빈 slice Accuracy는 결측으로 보존한다.

기존 train-quantile duration·confidence 진단과 4초 duration 경계 표를 따로 보존한다. HCE threshold는 기존 0.90, confidence는 max(prob0, prob1)이다.

In [8]:
predictions = comparison.prediction_frame(result59, result88, tracking)
assert len(predictions) == 177891
slices = comparison.paired_slice_table(predictions)
comparison_table = pd.DataFrame([report59, report88])
metric_names = ["train_accuracy", "internal_accuracy", "gap", "recall_speaker_0", "recall_speaker_1", "errors", "FP", "FN", "HCE", "HCE_FP", "HCE_FN"]
delta = {name: report88[name] - report59[name] for name in metric_names}
assert int(slices.iloc[0].net_correction) == report59["errors"] - report88["errors"]
legacy_duration_rows = []
for name, result in (("59D", result59), ("88D", result88)):
    for label, left, right in (("<0.5s", 0, .5), ("0.5-1s", .5, 1), ("1-2s", 1, 2), ("2-4s", 2, 4), (">=4s", 4, np.inf)):
        mask = (predictions.duration_sec >= left) & (predictions.duration_sec < right)
        correct = result.y_pred[mask] == result.y_true[mask]
        legacy_duration_rows.append({"representation": name, "slice": label, "n": int(mask.sum()), "accuracy": float(correct.mean()) if len(correct) else None, "errors": int((~correct).sum())})
legacy_duration_table = pd.DataFrame(legacy_duration_rows)
display(comparison_table.loc[:, ["representation", *metric_names]])
display(pd.DataFrame([delta]))
display(slices)
display(legacy_duration_table)
for name, diagnostics in (("59D", diagnostics59), ("88D", diagnostics88)):
    print(name, "existing project diagnostics")
    display(diagnostics["duration_summary"])
    display(diagnostics["confidence_summary"])

,representation,train_accuracy,internal_accuracy,gap,recall_speaker_0,recall_speaker_1,errors,FP,FN,HCE,HCE_FP,HCE_FN
0,AF10_AF11_COMBINATION,0.875788,0.87376,0.002028,0.846680,0.898736,22457,13086,9371,2378,1197,1181
1,EGEMAPSV02_FUNCTIONALS_88D,0.866647,0.86399,0.002658,0.823646,0.901199,24195,15052,9143,2733,1414,1319


,train_accuracy,internal_accuracy,gap,recall_speaker_0,recall_speaker_1,errors,FP,FN,HCE,HCE_FP,HCE_FN
0,-0.00914,-0.00977,0.00063,-0.023034,0.002464,1738,1966,-228,355,217,138


,slice,n,both_correct,both_wrong,correction,regression,net_correction,correct_59d,errors_59d,accuracy_59d,correct_88d,errors_88d,accuracy_88d
0,all,177891,146653,15414,7043,8781,-1738,155434,22457,0.873760,153696,24195,0.863990
1,speaker0,85351,67495,10282,2804,4770,-1966,72265,13086,0.846680,70299,15052,0.823646
2,speaker1,92540,79158,5132,4239,4011,228,83169,9371,0.898736,83397,9143,0.901199
3,overlap,78581,60580,8654,4323,5024,-701,65604,12977,0.834858,64903,13678,0.825938
4,no_overlap,99310,86073,6760,2720,3757,-1037,89830,9480,0.904541,88793,10517,0.894099
5,duration_<0.5s,14076,11362,1112,699,903,-204,12265,1811,0.871341,12061,2015,0.856849
6,speaker0_duration_<0.5s,6728,5478,431,362,457,-95,5935,793,0.882134,5840,888,0.868014
7,speaker1_duration_<0.5s,7348,5884,681,337,446,-109,6330,1018,0.861459,6221,1127,0.846625
8,duration_0.5-1s,43199,32361,4579,2989,3270,-281,35631,7568,0.824811,35350,7849,0.818306
9,speaker0_duration_0.5-1s,19593,13976,2705,882,2030,-1148,16006,3587,0.816924,14858,4735,0.758332


,representation,slice,n,accuracy,errors
0,59D,<0.5s,14076,0.871341,1811
1,59D,0.5-1s,43199,0.824811,7568
2,59D,1-2s,58320,0.853086,8568
3,59D,2-4s,42240,0.913778,3642
4,59D,>=4s,20056,0.956721,868
5,88D,<0.5s,14076,0.856849,2015
6,88D,0.5-1s,43199,0.818306,7849
7,88D,1-2s,58320,0.841427,9248
8,88D,2-4s,42240,0.904687,4026
9,88D,>=4s,20056,0.947298,1057


59D existing project diagnostics


,slice,count,accuracy
0,q25_q75,88589,0.856562
1,q75_q95,35967,0.933383
2,q05_q25,35746,0.832737
3,q00_q05,8687,0.874986
4,q95_q100,8902,0.967535


,slice,count,accuracy
0,>=0.90,112228,0.978811
1,0.70-0.80,12414,0.749960
2,0.80-0.90,18093,0.857956
3,<0.60,23741,0.554652
4,0.60-0.70,11415,0.664301


88D existing project diagnostics


,slice,count,accuracy
0,q25_q75,88589,0.846143
1,q75_q95,35967,0.924097
2,q05_q25,35746,0.824987
3,q00_q05,8687,0.861517
4,q95_q100,8902,0.957762


,slice,count,accuracy
0,>=0.90,106968,0.974450
1,0.80-0.90,20941,0.852252
2,0.70-0.80,14764,0.745394
3,<0.60,19095,0.541398
4,0.60-0.70,16123,0.637040


## 8. Save independent result artifacts

새 ID의 fresh directory에 저장한다. 기존 artifact는 덮어쓰지 않는다. JSON은 환경·source·feature schema·split·model/scaler·모든 지표와 진단을 포함한다. parquet는 두 모델의 동일 row prediction·probability·confidence와 diagnostic column만 포함한다.

In [9]:
chain_metadata = json.loads((CHAIN_ROOT / "chain.json").read_text(encoding="utf-8"))
summary = {
    "experiment_id": EXPERIMENT_ID, "status": "COMPARISON_COMPLETE", "environment": ENVIRONMENT,
    "reference_definition": {"accuracy": comparison.CLEAN_ACCURACY, "confusion_matrix": comparison.CLEAN_CM, "choice": "user-selected clean chain"},
    "source59": {"chain_path": CHAIN_ROOT, "chain_sha256": comparison.file_sha256(CHAIN_ROOT / "chain.json"), "metadata": chain_metadata, "feature_names": clean.feature_names},
    "source88": {"cache_path": EGEMAPS_PATH, "cache_sha256": comparison.file_sha256(EGEMAPS_PATH), "metadata": egemaps.metadata, "feature_names": egemaps.feature_names},
    "alignment": alignment, "split": split_audit, "seed": 42,
    "model_config": dict(result59.model_config), "scaler_fit_scope": "internal_train_only", "threshold": .5,
    "metrics59": report59, "metrics88": report88, "delta_88d_minus_59d": delta,
    "reproduction": reproduction, "paired_slices": slices.to_dict(orient="records"),
    "legacy_duration_slices": legacy_duration_table.to_dict(orient="records"),
    "project_diagnostics": {name: {
        "duration_boundaries_ms": diagnostic["duration_boundaries_ms"],
        "duration_summary": diagnostic["duration_summary"].to_dict(orient="records"),
        "confidence_summary": diagnostic["confidence_summary"].to_dict(orient="records"),
        "overlap_summary": diagnostic["overlap_summary"].to_dict(orient="records"),
    } for name, diagnostic in (("59D", diagnostics59), ("88D", diagnostics88))},
    "model_inputs": {"59D": clean.feature_names, "88D": egemaps.feature_names},
    "diagnostic_columns_only": ["record_key", "call_id", "duration_sec", "overlap"],
    "leakage_checks": {"official_validation_used": False, "validation_in_gradient": False, "call_overlap": split_audit["call_overlap"], "scaler_fit_rows": 711467},
    "limitations": ["Historical row prediction equality unavailable", "Clean raw-content proof retained from chain metadata", "Cross-platform overlap/approval retained from metadata", "Matrix bytes are not peak memory"],
}
comparison.save_artifacts(OUTPUT_ROOT / "comparison", summary, predictions, comparison_table, slices)
print("Saved:", OUTPUT_ROOT)

Saved: C:\Users\YW\Study\02_Projects\DCC_2026\prelim\data\processed\mission2_standard_representation_comparison\clean59d_egemaps88d_20261004T190153_4dbd7177


## 9. Final summary / 사용자 판단

관찰만 작성하며 bottleneck을 자동 확정하거나 canonical을 승격하지 않는다.

- Integrity / Alignment: checkpoint 2–4
- 59D Reproduction / eGeMAPS: Train/Internal Accuracy, gap, Recall, CM, HCE
- Delta / Paired change: checkpoint 7과 paired_slices.csv
- Duration / Overlap / Weak slice: slice 표와 summary JSON
- Fit / Cost: n_iter, warning, loss, fit/inference time, matrix bytes
- Leakage: Training 내부 group split·train-only scaler·acoustic-only input
- Interpretation: 실행 후 사용자가 관찰을 작성
- Next Decision: 다음 판단을 바꾸는 evidence를 사용자가 정리

실험로그·roadmap은 여기서 수정하지 않는다.

In [10]:
display(comparison_table.loc[:, ["representation", "train_accuracy", "internal_accuracy", "gap", "recall_speaker_0", "recall_speaker_1", "confusion_matrix", "HCE"]])
display(pd.DataFrame([delta]))
display(slices.loc[slices["slice"].isin(["all", "overlap", "no_overlap", "speaker0_0.5-1s_overlap", "speaker0_1-2s_overlap"])])
display(comparison_table.loc[:, ["representation", "n_iter", "convergence_warning", "final_loss", "fit_seconds", "validation_inference_seconds", "train_inference_seconds", "scaled_matrix_bytes"]])
print("Artifacts:", OUTPUT_ROOT)
print("Interpretation / next decision: user review pending")

,representation,train_accuracy,internal_accuracy,gap,recall_speaker_0,recall_speaker_1,confusion_matrix,HCE
0,AF10_AF11_COMBINATION,0.875788,0.87376,0.002028,0.846680,0.898736,"[[72265, 13086], [9371, 83169]]",2378
1,EGEMAPSV02_FUNCTIONALS_88D,0.866647,0.86399,0.002658,0.823646,0.901199,"[[70299, 15052], [9143, 83397]]",2733


,train_accuracy,internal_accuracy,gap,recall_speaker_0,recall_speaker_1,errors,FP,FN,HCE,HCE_FP,HCE_FN
0,-0.00914,-0.00977,0.00063,-0.023034,0.002464,1738,1966,-228,355,217,138


,slice,n,both_correct,both_wrong,correction,regression,net_correction,correct_59d,errors_59d,accuracy_59d,correct_88d,errors_88d,accuracy_88d
0,all,177891,146653,15414,7043,8781,-1738,155434,22457,0.873760,153696,24195,0.863990
3,overlap,78581,60580,8654,4323,5024,-701,65604,12977,0.834858,64903,13678,0.825938
4,no_overlap,99310,86073,6760,2720,3757,-1037,89830,9480,0.904541,88793,10517,0.894099
20,speaker0_0.5-1s_overlap,7976,4885,1504,528,1059,-531,5944,2032,0.745236,5413,2563,0.678661
21,speaker0_1-2s_overlap,13517,9601,2397,617,902,-285,10503,3014,0.777022,10218,3299,0.755937


,representation,n_iter,convergence_warning,final_loss,fit_seconds,validation_inference_seconds,train_inference_seconds,scaled_matrix_bytes
0,AF10_AF11_COMBINATION,100,True,0.266970,189.472715,0.119767,0.319432,209888488
1,EGEMAPSV02_FUNCTIONALS_88D,100,True,0.289571,186.145962,0.404899,0.378094,313054016


Artifacts: C:\Users\YW\Study\02_Projects\DCC_2026\prelim\data\processed\mission2_standard_representation_comparison\clean59d_egemaps88d_20261004T190153_4dbd7177
Interpretation / next decision: user review pending
